# Week 2 - Day 3
## Task 1 - Imputing and Encoding

### Aim
To handle missing values using SimpleImputer and KNNImputer, and encode categorical data using OneHotEncoder, OrdinalEncoder and LabelEncoder.


In [1]:
import pandas as pd

from sklearn.impute import SimpleImputer, KNNImputer

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    LabelEncoder
)

In [2]:
df = pd.DataFrame({
    'age': [25, None, 40, 35, None],
    'income': [30000, 45000, None, 80000, 52000],
    'city': ['Chennai', 'Mumbai', None, 'Chennai', 'Delhi'],
    'size': ['S', 'L', 'M', 'S', 'L']
})

print("Original DataFrame:")
print(df)

Original DataFrame:
    age   income     city size
0  25.0  30000.0  Chennai    S
1   NaN  45000.0   Mumbai    L
2  40.0      NaN      NaN    M
3  35.0  80000.0  Chennai    S
4   NaN  52000.0    Delhi    L


In [3]:
simple_df = df.copy()

age_imputer = SimpleImputer(strategy='median')

simple_df[['age']] = age_imputer.fit_transform(
    simple_df[['age']]
)

print("After SimpleImputer (Median) on Age:")
print(simple_df)


After SimpleImputer (Median) on Age:
    age   income     city size
0  25.0  30000.0  Chennai    S
1  35.0  45000.0   Mumbai    L
2  40.0      NaN      NaN    M
3  35.0  80000.0  Chennai    S
4  35.0  52000.0    Delhi    L


In [4]:
city_imputer = SimpleImputer(strategy='most_frequent')

simple_df[['city']] = city_imputer.fit_transform(
    simple_df[['city']]
)

print("After SimpleImputer (Most Frequent City):")
print(simple_df)

After SimpleImputer (Most Frequent City):
    age   income     city size
0  25.0  30000.0  Chennai    S
1  35.0  45000.0   Mumbai    L
2  40.0      NaN  Chennai    M
3  35.0  80000.0  Chennai    S
4  35.0  52000.0    Delhi    L


In [5]:
knn_df = df.copy()

knn_imputer = KNNImputer(n_neighbors=2)

knn_df[['age', 'income']] = knn_imputer.fit_transform(
    knn_df[['age', 'income']]
)

print("After KNNImputer:")
print(knn_df[['age', 'income']])


After KNNImputer:
    age   income
0  25.0  30000.0
1  30.0  45000.0
2  40.0  55000.0
3  35.0  80000.0
4  30.0  52000.0


In [6]:
comparison = pd.DataFrame({
    'Original Age': df['age'],
    'SimpleImputer Age': simple_df['age'],
    'KNNImputer Age': knn_df['age'],
    'Original Income': df['income'],
    'KNNImputer Income': knn_df['income']
})

print("SimpleImputer vs KNNImputer:")
print(comparison)

SimpleImputer vs KNNImputer:
   Original Age  SimpleImputer Age  KNNImputer Age  Original Income  \
0          25.0               25.0            25.0          30000.0   
1           NaN               35.0            30.0          45000.0   
2          40.0               40.0            40.0              NaN   
3          35.0               35.0            35.0          80000.0   
4           NaN               35.0            30.0          52000.0   

   KNNImputer Income  
0            30000.0  
1            45000.0  
2            55000.0  
3            80000.0  
4            52000.0  


In [7]:
city_encoder = OneHotEncoder(sparse_output=False)

city_encoded = city_encoder.fit_transform(
    simple_df[['city']]
)

city_df = pd.DataFrame(
    city_encoded,
    columns=city_encoder.get_feature_names_out(['city'])
)

print("After OneHotEncoder on City:")
print(city_df)

After OneHotEncoder on City:
   city_Chennai  city_Delhi  city_Mumbai
0           1.0         0.0          0.0
1           0.0         0.0          1.0
2           1.0         0.0          0.0
3           1.0         0.0          0.0
4           0.0         1.0          0.0


In [8]:
size_encoder = OrdinalEncoder(
    categories=[['S', 'M', 'L']]
)

size_encoded = size_encoder.fit_transform(
    simple_df[['size']]
)

size_df = pd.DataFrame(
    size_encoded.astype(int),
    columns=['size_encoded']
)

print("After OrdinalEncoder on Size:")
print(size_df)


After OrdinalEncoder on Size:
   size_encoded
0             0
1             2
2             1
3             0
4             2


In [9]:
target = ['Yes', 'No', 'Yes', 'No', 'Yes']

label_encoder = LabelEncoder()

target_encoded = label_encoder.fit_transform(target)

print("Original Target:", target)
print("Encoded Target:", target_encoded)
print("Classes:", label_encoder.classes_)


Original Target: ['Yes', 'No', 'Yes', 'No', 'Yes']
Encoded Target: [1 0 1 0 1]
Classes: ['No' 'Yes']


In [10]:
final_df = pd.concat([
    simple_df[['age']].reset_index(drop=True),
    knn_df[['income']].reset_index(drop=True),
    city_df.reset_index(drop=True),
    size_df.reset_index(drop=True)
], axis=1)

print("Final Encoded DataFrame:")
print(final_df)


Final Encoded DataFrame:
    age   income  city_Chennai  city_Delhi  city_Mumbai  size_encoded
0  25.0  30000.0           1.0         0.0          0.0             0
1  35.0  45000.0           0.0         0.0          1.0             2
2  40.0  55000.0           1.0         0.0          0.0             1
3  35.0  80000.0           1.0         0.0          0.0             0
4  35.0  52000.0           0.0         1.0          0.0             2


## Conclusion

1. Created a DataFrame containing missing values.
2. Used SimpleImputer to fill missing age values with the median and missing city values with the most frequent city.
3. Applied KNNImputer to numeric columns and compared its results with SimpleImputer.
4. Used OneHotEncoder to encode city and OrdinalEncoder to encode size.
5. Demonstrated LabelEncoder on a sample target column.
6. Created the final encoded DataFrame without missing values.

Thus, missing values were handled and categorical data was successfully encoded using different preprocessing techniques.

# WEEK 2 - DAY 3
# TASK 2 - SCALING

In [11]:
import pandas as pd

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# ------------------------------------------------
# PART 1: STANDARD SCALER
# ------------------------------------------------

print("PART 1: STANDARD SCALER")

numeric_data = final_df[['age', 'income']]

print("\nOriginal Data:")
print(numeric_data)

standard_scaler = StandardScaler()

standard_scaled = standard_scaler.fit_transform(numeric_data)

standard_df = pd.DataFrame(
    standard_scaled,
    columns=['age', 'income']
)

print("\nAfter StandardScaler:")
print(standard_df.round(4))

print("\nMean:")
print(standard_df.mean().round(4))

print("\nStandard Deviation:")
print(standard_df.std(ddof=0).round(4))

# ------------------------------------------------
# PART 2: MINMAX SCALER
# ------------------------------------------------

print("\nPART 2: MINMAX SCALER")

minmax_scaler = MinMaxScaler()

minmax_scaled = minmax_scaler.fit_transform(numeric_data)

minmax_df = pd.DataFrame(
    minmax_scaled,
    columns=['age', 'income']
)

print("\nAfter MinMaxScaler:")
print(minmax_df.round(4))

print("\nMinimum Values:")
print(minmax_df.min())

print("\nMaximum Values:")
print(minmax_df.max())

# ------------------------------------------------
# PART 3: KNN WITH AND WITHOUT SCALING
# ------------------------------------------------

print("\nPART 3: KNN CLASSIFIER")

data = load_breast_cancer()

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# KNN without scaling
knn_without = KNeighborsClassifier(n_neighbors=7)

knn_without.fit(X_train, y_train)

pred_without = knn_without.predict(X_test)

accuracy_without = accuracy_score(y_test, pred_without)

# StandardScaler: fit only on training data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

# KNN with scaling
knn_with = KNeighborsClassifier(n_neighbors=7)

knn_with.fit(X_train_scaled, y_train)

pred_with = knn_with.predict(X_test_scaled)

accuracy_with = accuracy_score(y_test, pred_with)

print("\nKNN without scaling:", round(accuracy_without, 4))
print("KNN with scaling:", round(accuracy_with, 4))

# ------------------------------------------------
# PART 4: DECISION TREE WITH AND WITHOUT SCALING
# ------------------------------------------------

print("\nPART 4: DECISION TREE CLASSIFIER")

# Decision Tree without scaling
tree_without = DecisionTreeClassifier(random_state=42)

tree_without.fit(X_train, y_train)

tree_pred_without = tree_without.predict(X_test)

tree_accuracy_without = accuracy_score(
    y_test, tree_pred_without
)

# Decision Tree with scaling
tree_with = DecisionTreeClassifier(random_state=42)

tree_with.fit(X_train_scaled, y_train)

tree_pred_with = tree_with.predict(X_test_scaled)

tree_accuracy_with = accuracy_score(
    y_test, tree_pred_with
)

print("\nDecision Tree without scaling:",
      round(tree_accuracy_without, 4))

print("Decision Tree with scaling:",
      round(tree_accuracy_with, 4))

if tree_accuracy_without == tree_accuracy_with:
    print("Decision Tree accuracy is the same in both cases.")
else:
    print("Decision Tree accuracy differs in this experiment.")

# ------------------------------------------------
# PART 5: FINAL COMPARISON
# ------------------------------------------------

print("\nPART 5: FINAL MODEL COMPARISON")

comparison = pd.DataFrame({
    'Model': ['KNN', 'Decision Tree'],
    'Without Scaling': [
        accuracy_without,
        tree_accuracy_without
    ],
    'With Scaling': [
        accuracy_with,
        tree_accuracy_with
    ]
})

print(comparison.round(4))

PART 1: STANDARD SCALER

Original Data:
    age   income
0  25.0  30000.0
1  35.0  45000.0
2  40.0  55000.0
3  35.0  80000.0
4  35.0  52000.0

After StandardScaler:
      age  income
0 -1.8371 -1.3759
1  0.2041 -0.4545
2  1.2247  0.1597
3  0.2041  1.6953
4  0.2041 -0.0246

Mean:
age       0.0
income   -0.0
dtype: float64

Standard Deviation:
age       1.0
income    1.0
dtype: float64

PART 2: MINMAX SCALER

After MinMaxScaler:
      age  income
0  0.0000    0.00
1  0.6667    0.30
2  1.0000    0.50
3  0.6667    1.00
4  0.6667    0.44

Minimum Values:
age       0.0
income    0.0
dtype: float64

Maximum Values:
age       1.0
income    1.0
dtype: float64

PART 3: KNN CLASSIFIER

KNN without scaling: 0.9298
KNN with scaling: 0.9737

PART 4: DECISION TREE CLASSIFIER

Decision Tree without scaling: 0.9123
Decision Tree with scaling: 0.9123
Decision Tree accuracy is the same in both cases.

PART 5: FINAL MODEL COMPARISON
           Model  Without Scaling  With Scaling
0            KNN         

Conclusion

StandardScaler transforms the data to have a mean of approximately 0 and a standard deviation of 1. MinMaxScaler scales the values between 0 and 1. KNN performs better after scaling because it is distance-based. Decision Tree accuracy remains the same because it does not depend on feature scaling.

# Bonus Challenge - RobustScaler

In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, RobustScaler

data = pd.DataFrame({
    'age': [25, 30, 35, 40, 45, 100],
    'income': [30000, 35000, 40000, 45000, 50000, 200000]
})

print("Original Data:")
print(data)

standard_scaler = StandardScaler()
standard_scaled = standard_scaler.fit_transform(data)

standard_df = pd.DataFrame(
    standard_scaled,
    columns=data.columns
)

print("\nAfter StandardScaler:")
print(standard_df.round(2))

robust_scaler = RobustScaler()
robust_scaled = robust_scaler.fit_transform(data)

robust_df = pd.DataFrame(
    robust_scaled,
    columns=data.columns
)

print("\nAfter RobustScaler:")
print(robust_df.round(2))

print("\nComparison:")
comparison = pd.DataFrame({
    'Original Age': data['age'],
    'StandardScaler Age': standard_df['age'],
    'RobustScaler Age': robust_df['age']
})

print(comparison.round(2))

print("\nConclusion:")
print("RobustScaler is less affected by outliers than StandardScaler.")

Original Data:
   age  income
0   25   30000
1   30   35000
2   35   40000
3   40   45000
4   45   50000
5  100  200000

After StandardScaler:
    age  income
0 -0.83   -0.61
1 -0.63   -0.53
2 -0.43   -0.44
3 -0.23   -0.36
4 -0.03   -0.28
5  2.16    2.22

After RobustScaler:
   age  income
0 -1.0    -1.0
1 -0.6    -0.6
2 -0.2    -0.2
3  0.2     0.2
4  0.6     0.6
5  5.0    12.6

Comparison:
   Original Age  StandardScaler Age  RobustScaler Age
0            25               -0.83              -1.0
1            30               -0.63              -0.6
2            35               -0.43              -0.2
3            40               -0.23               0.2
4            45               -0.03               0.6
5           100                2.16               5.0

Conclusion:
RobustScaler is less affected by outliers than StandardScaler.
